In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
current_directory= Path.cwd()
project_root=current_directory.parent
actual_file=project_root/"data"/"raw"/"datos_sucios_hito1.csv"

if not actual_file.exists():
    raise FileNotFoundError("Falta el insumo de trabajo. Debe nombrarlo como \"datos_sucios_hito1.csv\" y guardarlo en data/raw/")
df_contacts=pd.read_csv(actual_file)

In [ ]:
nan_count=df_contacts["Contacto"].isna().sum()
empty_count=(df_contacts["Contacto"].str.strip()=="").sum()
only_number_count=df_contacts["Contacto"].str.replace(" ","").str.replace(".","").str.isdigit().sum()
anythin_else_count=(df_contacts["Contacto"].str.contains(" - | / |Cel|Correo|@", na=False).sum())
print(only_number_count+nan_count+empty_count+anythin_else_count)

In [ ]:
def split_phone_email(text):
    '''
    Separa los datos teléfono/correo según las condiciones impuestas y genera una lista con los mismos o la lista con el valor "Sin dato"
    si text es vacío o NaN.

    Args:
        text (str): cadena de texto. Representa los valores de la columna "Contacto" del DataFrame original.
    
    Returns:
        list: lista. Generada con el método split que contiene los datos teléfono/correo separados o la lista con el valor
        "Sin dato" si text es vacío o NaN.
    
    Raises:
        AttributeError: si text es un tipo de valor diferente a str.
    '''

    if pd.isna(text) or not text:
        return ["Sin dato"]
    elif text.startswith("Cel:"):
        return text.replace(" ","").replace("Cel:","").split("Correo:")
    elif text.startswith("Correo:"):
        return text.replace(" ","").replace("Correo:","").split("Cel:")
    elif "/" in text:
        return text.replace(" ","").split("/")
    elif "-" in text:
        return text.replace(" ","").split("-")
    else:
        return text.split()
phone_email=df_contacts["Contacto"].apply(split_phone_email).tolist() #Se convierte a lista porque este resultado se usará con un ciclo for no vectorizable, pues contiene valores no uniformes en longitud, pues row puede contener 1 o 2 elementos según el caso.

In [ ]:
def is_phone_email(row):
    '''
    Evalúa cada valor y determina si se trata de teléfono o de correo según las condiciones impuestas.

    Args:
        row (list): lista. Contiene los valores generados por la función split_phone_email.
    
    Returns:
        tuple: tupla. Asigna una etiqueta "phone" o "email" a cada valor de la tupla en ese orden, según las condiciones impuestas, o "Sin dato"
        para el valor (o ambos) faltante.
    
    Raises:
        AttributeError: si row contiene valores que no sean tipo str.
    
    Notes:
        La función depende de que row contenga uno o dos elementos de tipo str.
    '''
    if len(row)==1:
        if row[0].replace(" ","").isdigit():
            phone=row[0]
            email="Sin dato"
        elif "@" in row[0]:
            phone="Sin dato"
            email=row[0]
        elif row[0].replace("@","").replace("_","").replace(".","").isalnum():
            phone="Sin dato"
            email=row[0]
        else:
            phone="Sin dato"
            email="Sin dato"
    else:
        if row[0].replace(" ","").isdigit():
            phone=row[0]
            email=row[1]
        elif "@" in row[0]:
            phone=row[1]
            email=row[0]
        elif row[0].replace("@","").replace("_","").replace(".","").isalnum():
            phone=row[1]
            email=row[0]
        else:
            phone="Sin dato"
            email="Sin dato"

    return phone, email

results=[]
for row in phone_email:
    results.append(is_phone_email(row))

In [ ]:
phone_list=[phone for phone,email in results]
email_list=[email for phone,email in results]

if "Teléfono" not in df_contacts.columns:
    df_contacts.insert(loc=4,column="Teléfono",value=phone_list)

if "Correo" not in df_contacts.columns:
    df_contacts.insert(loc=5,column="Correo",value=email_list)

if "Contacto" in df_contacts.columns:
    df_contacts.drop(columns=["Contacto"],inplace=True)

df_contacts